In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

SILVER_SCHEMA = "supplysage_silver"

VALIDATION_RUN_ID = f"silver_relationship_validation_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "14_silver_relationship_validation"

required_silver_tables = [
    "silver_m5_calendar",
    "silver_m5_sell_prices",
    "silver_m5_sales_daily",
    "silver_retail_inventory",
    "silver_dataco_orders_shipments",
    "silver_dataco_field_dictionary",
    "silver_suppliers",
    "silver_supplier_aliases",
    "silver_supplier_sku_map",
    "silver_alternate_suppliers",
    "silver_supplier_scorecards",
    "silver_purchase_orders",
    "silver_shipment_routes",
    "silver_product_crosswalk",
    "silver_external_risk_events",
    "silver_external_evidence_documents",
    "silver_transform_validation_results"
]

print("VALIDATION_RUN_ID:", VALIDATION_RUN_ID)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

table_check_rows = []

for table_name in required_silver_tables:
    full_table_name = f"{SILVER_SCHEMA}.{table_name}"

    try:
        df = spark.table(full_table_name)
        row_count = df.count()
        status = "FOUND"
        issue_detail = None

        print(f"{table_name}: {row_count} rows")

    except Exception as e:
        row_count = 0
        status = "MISSING"
        issue_detail = str(e)[:500]

        print(f"{table_name}: MISSING")

    table_check_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "silver_table": table_name,
        "validation_check": "silver_table_exists",
        "expected_value": "table exists",
        "actual_value": str(row_count),
        "status": "PASS" if status == "FOUND" and row_count > 0 else "FAIL",
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


relationship_validation_schema = T.StructType([
    T.StructField("validation_run_id", T.StringType(), True),
    T.StructField("silver_table", T.StringType(), True),
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True),
    T.StructField("validated_at", T.StringType(), True),
    T.StructField("created_by", T.StringType(), True),
    T.StructField("notebook_name", T.StringType(), True)
])

table_existence_validation_df = spark.createDataFrame(
    table_check_rows,
    schema=relationship_validation_schema
)

display(table_existence_validation_df.orderBy("status", "silver_table"))

missing_or_empty_count = table_existence_validation_df.filter(F.col("status") == "FAIL").count()

print("Missing or empty Silver tables:", missing_or_empty_count)

if missing_or_empty_count == 0:
    print("✅ Chunk 1 PASSED: All required Silver tables exist and contain rows.")
else:
    print("❌ Chunk 1 FAILED: Some Silver tables are missing or empty.")

In [0]:
# Notebook 14 Fix: Add missing supplier/internal SKUs to silver_product_crosswalk

from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

SILVER_SCHEMA = "supplysage_silver"

silver_supplier_sku_map_df = spark.table(f"{SILVER_SCHEMA}.silver_supplier_sku_map")
silver_purchase_orders_df = spark.table(f"{SILVER_SCHEMA}.silver_purchase_orders")
silver_product_crosswalk_df = spark.table(f"{SILVER_SCHEMA}.silver_product_crosswalk")

# 1. Collect all supplier-side SKU IDs that should exist as canonical SKUs.
supplier_side_skus_df = (
    silver_supplier_sku_map_df
    .select(
        F.col("sku_id").alias("canonical_sku_id"),
        F.col("synthetic_batch_id"),
        F.col("is_synthetic")
    )
    .unionByName(
        silver_purchase_orders_df
        .select(
            F.col("sku_id").alias("canonical_sku_id"),
            F.col("synthetic_batch_id"),
            F.col("is_synthetic")
        )
    )
    .filter(F.col("canonical_sku_id").isNotNull())
    .dropDuplicates(["canonical_sku_id"])
)

# 2. Find supplier-side SKUs missing from the product crosswalk.
missing_supplier_skus_df = (
    supplier_side_skus_df
    .join(
        silver_product_crosswalk_df
        .select("canonical_sku_id")
        .distinct(),
        on="canonical_sku_id",
        how="left_anti"
    )
)

missing_supplier_sku_count = missing_supplier_skus_df.count()

print("Missing supplier-side SKUs to add to silver_product_crosswalk:", missing_supplier_sku_count)

# 3. Create self-mapping rows for missing supplier SKUs.
missing_crosswalk_rows_df = (
    missing_supplier_skus_df
    .select(
        F.col("canonical_sku_id"),
        F.lit("supplier_internal").alias("source_system"),
        F.col("canonical_sku_id").alias("source_product_id"),
        F.col("canonical_sku_id").alias("source_product_name"),
        F.lit("supplier_internal").alias("source_category"),
        F.lit("supplier_internal").alias("source_department"),
        F.lit(1.0).cast("double").alias("confidence"),
        F.lit("supplier_sku_self_mapping_added_for_relationship_integrity").alias("mapping_rule"),
        F.col("synthetic_batch_id"),
        F.col("is_synthetic")
    )
    .withColumn(
        "product_crosswalk_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("canonical_sku_id"),
                F.col("source_system"),
                F.col("source_product_id")
            ),
            256
        )
    )
    .withColumn("silver_transform_run_id", F.lit("relationship_fix_supplier_sku_crosswalk"))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit("Vigya"))
    .withColumn("silver_notebook_name", F.lit("14_silver_relationship_validation"))
)

# 4. Union existing crosswalk with new self-mapping rows.
fixed_product_crosswalk_df = (
    silver_product_crosswalk_df
    .unionByName(missing_crosswalk_rows_df)
    .dropDuplicates(["canonical_sku_id", "source_system", "source_product_id"])
)

fixed_crosswalk_count = fixed_product_crosswalk_df.count()

print("Original silver_product_crosswalk rows:", silver_product_crosswalk_df.count())
print("Fixed silver_product_crosswalk rows:", fixed_crosswalk_count)

# 5. Overwrite silver_product_crosswalk with the fixed version.
(
    fixed_product_crosswalk_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.silver_product_crosswalk")
)

print("✅ Fixed silver_product_crosswalk with supplier_internal self-mappings.")

In [0]:
 # Notebook 14, Chunk 2: Validate core Silver relationships

relationship_rows = []

silver_suppliers_df = spark.table(f"{SILVER_SCHEMA}.silver_suppliers")
silver_supplier_aliases_df = spark.table(f"{SILVER_SCHEMA}.silver_supplier_aliases")
silver_supplier_sku_map_df = spark.table(f"{SILVER_SCHEMA}.silver_supplier_sku_map")
silver_alternate_suppliers_df = spark.table(f"{SILVER_SCHEMA}.silver_alternate_suppliers")
silver_supplier_scorecards_df = spark.table(f"{SILVER_SCHEMA}.silver_supplier_scorecards")
silver_purchase_orders_df = spark.table(f"{SILVER_SCHEMA}.silver_purchase_orders")
silver_shipment_routes_df = spark.table(f"{SILVER_SCHEMA}.silver_shipment_routes")
silver_product_crosswalk_df = spark.table(f"{SILVER_SCHEMA}.silver_product_crosswalk")
silver_external_events_df = spark.table(f"{SILVER_SCHEMA}.silver_external_risk_events")
silver_external_documents_df = spark.table(f"{SILVER_SCHEMA}.silver_external_evidence_documents")
silver_calendar_df = spark.table(f"{SILVER_SCHEMA}.silver_m5_calendar")
silver_sales_daily_df = spark.table(f"{SILVER_SCHEMA}.silver_m5_sales_daily")
silver_sell_prices_df = spark.table(f"{SILVER_SCHEMA}.silver_m5_sell_prices")
silver_inventory_df = spark.table(f"{SILVER_SCHEMA}.silver_retail_inventory")
silver_dataco_df = spark.table(f"{SILVER_SCHEMA}.silver_dataco_orders_shipments")


def add_relationship_check(silver_table, validation_check, expected_value, actual_value, status, issue_detail=None):
    relationship_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "silver_table": silver_table,
        "validation_check": validation_check,
        "expected_value": str(expected_value),
        "actual_value": str(actual_value),
        "status": status,
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


# 1. Supplier aliases should point to valid suppliers
alias_orphan_supplier_count = (
    silver_supplier_aliases_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

add_relationship_check(
    "silver_supplier_aliases",
    "alias_supplier_id_exists_in_suppliers",
    "0 orphan supplier ids",
    alias_orphan_supplier_count,
    "PASS" if alias_orphan_supplier_count == 0 else "FAIL",
    None if alias_orphan_supplier_count == 0 else "supplier_aliases contains supplier_id values missing from silver_suppliers."
)


# 2. Supplier SKU map should point to valid suppliers
sku_map_orphan_supplier_count = (
    silver_supplier_sku_map_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

add_relationship_check(
    "silver_supplier_sku_map",
    "sku_map_supplier_id_exists_in_suppliers",
    "0 orphan supplier ids",
    sku_map_orphan_supplier_count,
    "PASS" if sku_map_orphan_supplier_count == 0 else "FAIL",
    None if sku_map_orphan_supplier_count == 0 else "supplier_sku_map contains supplier_id values missing from silver_suppliers."
)


# 3. Supplier scorecards should point to valid suppliers
scorecard_orphan_supplier_count = (
    silver_supplier_scorecards_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

add_relationship_check(
    "silver_supplier_scorecards",
    "scorecard_supplier_id_exists_in_suppliers",
    "0 orphan supplier ids",
    scorecard_orphan_supplier_count,
    "PASS" if scorecard_orphan_supplier_count == 0 else "FAIL",
    None if scorecard_orphan_supplier_count == 0 else "supplier_scorecards contains supplier_id values missing from silver_suppliers."
)


# 4. Purchase orders should point to valid suppliers
po_orphan_supplier_count = (
    silver_purchase_orders_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

add_relationship_check(
    "silver_purchase_orders",
    "po_supplier_id_exists_in_suppliers",
    "0 orphan supplier ids",
    po_orphan_supplier_count,
    "PASS" if po_orphan_supplier_count == 0 else "FAIL",
    None if po_orphan_supplier_count == 0 else "purchase_orders contains supplier_id values missing from silver_suppliers."
)


# 5. Shipment routes should point to valid suppliers
route_orphan_supplier_count = (
    silver_shipment_routes_df
    .join(silver_suppliers_df.select("supplier_id"), on="supplier_id", how="left_anti")
    .count()
)

add_relationship_check(
    "silver_shipment_routes",
    "route_supplier_id_exists_in_suppliers",
    "0 orphan supplier ids",
    route_orphan_supplier_count,
    "PASS" if route_orphan_supplier_count == 0 else "FAIL",
    None if route_orphan_supplier_count == 0 else "shipment_routes contains supplier_id values missing from silver_suppliers."
)


# 6. Alternate supplier primary supplier should exist
alternate_primary_orphan_count = (
    silver_alternate_suppliers_df
    .join(
        silver_suppliers_df.select(F.col("supplier_id").alias("primary_supplier_id")),
        on="primary_supplier_id",
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_alternate_suppliers",
    "primary_supplier_id_exists_in_suppliers",
    "0 orphan primary supplier ids",
    alternate_primary_orphan_count,
    "PASS" if alternate_primary_orphan_count == 0 else "FAIL",
    None if alternate_primary_orphan_count == 0 else "alternate_suppliers contains primary_supplier_id values missing from silver_suppliers."
)


# 7. Alternate supplier alternate supplier should exist
alternate_supplier_orphan_count = (
    silver_alternate_suppliers_df
    .join(
        silver_suppliers_df.select(F.col("supplier_id").alias("alternate_supplier_id")),
        on="alternate_supplier_id",
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_alternate_suppliers",
    "alternate_supplier_id_exists_in_suppliers",
    "0 orphan alternate supplier ids",
    alternate_supplier_orphan_count,
    "PASS" if alternate_supplier_orphan_count == 0 else "FAIL",
    None if alternate_supplier_orphan_count == 0 else "alternate_suppliers contains alternate_supplier_id values missing from silver_suppliers."
)


# 8. Supplier SKU map SKU IDs should exist in product crosswalk canonical SKUs
sku_map_orphan_sku_count = (
    silver_supplier_sku_map_df
    .select("sku_id")
    .distinct()
    .join(
        silver_product_crosswalk_df.select(F.col("canonical_sku_id").alias("sku_id")).distinct(),
        on="sku_id",
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_supplier_sku_map",
    "sku_id_exists_in_product_crosswalk",
    "0 orphan sku ids",
    sku_map_orphan_sku_count,
    "PASS" if sku_map_orphan_sku_count == 0 else "FAIL",
    None if sku_map_orphan_sku_count == 0 else "supplier_sku_map contains sku_id values missing from silver_product_crosswalk canonical_sku_id."
)


# 9. Purchase order SKU IDs should exist in product crosswalk canonical SKUs
po_orphan_sku_count = (
    silver_purchase_orders_df
    .select("sku_id")
    .distinct()
    .join(
        silver_product_crosswalk_df.select(F.col("canonical_sku_id").alias("sku_id")).distinct(),
        on="sku_id",
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_purchase_orders",
    "po_sku_id_exists_in_product_crosswalk",
    "0 orphan sku ids",
    po_orphan_sku_count,
    "PASS" if po_orphan_sku_count == 0 else "FAIL",
    None if po_orphan_sku_count == 0 else "purchase_orders contains sku_id values missing from silver_product_crosswalk canonical_sku_id."
)


# 10. External events should link back to evidence documents
external_event_document_orphan_count = (
    silver_external_events_df
    .select("api_run_id", "source_payload_hash")
    .distinct()
    .join(
        silver_external_documents_df
        .select(
            "api_run_id",
            F.col("payload_hash").alias("source_payload_hash")
        )
        .distinct(),
        on=["api_run_id", "source_payload_hash"],
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_external_risk_events",
    "events_link_to_evidence_documents",
    "0 orphan event document references",
    external_event_document_orphan_count,
    "PASS" if external_event_document_orphan_count == 0 else "FAIL",
    None if external_event_document_orphan_count == 0 else "Some external events do not link back to silver_external_evidence_documents."
)


# 11. Sales daily should have complete calendar dates already joined
sales_missing_calendar_count = (
    silver_sales_daily_df
    .filter(
        F.col("calendar_date").isNull()
        | F.col("calendar_date_id").isNull()
        | F.col("wm_year_week").isNull()
    )
    .count()
)

add_relationship_check(
    "silver_m5_sales_daily",
    "sales_daily_calendar_join_complete",
    "0 missing calendar rows",
    sales_missing_calendar_count,
    "PASS" if sales_missing_calendar_count == 0 else "FAIL",
    None if sales_missing_calendar_count == 0 else "silver_m5_sales_daily has missing calendar fields."
)


# 12. Sell price weeks should exist in calendar weeks
sell_price_orphan_week_count = (
    silver_sell_prices_df
    .select("wm_year_week")
    .distinct()
    .join(
        silver_calendar_df.select("wm_year_week").distinct(),
        on="wm_year_week",
        how="left_anti"
    )
    .count()
)

add_relationship_check(
    "silver_m5_sell_prices",
    "sell_price_weeks_exist_in_calendar",
    "0 orphan wm_year_week values",
    sell_price_orphan_week_count,
    "PASS" if sell_price_orphan_week_count == 0 else "FAIL",
    None if sell_price_orphan_week_count == 0 else "sell_prices contains wm_year_week values missing from silver_m5_calendar."
)


relationship_core_validation_df = spark.createDataFrame(
    relationship_rows,
    schema=relationship_validation_schema
)

display(relationship_core_validation_df.orderBy("status", "silver_table", "validation_check"))

core_fail_count = relationship_core_validation_df.filter(F.col("status") == "FAIL").count()

print("Core Silver relationship validation failures:", core_fail_count)

In [0]:
display(
    relationship_core_validation_df
    .filter(F.col("status") == "FAIL")
    .select(
        "silver_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "issue_detail"
    )
)

In [0]:
# Notebook 14, Chunk 3: Save relationship validation results

all_relationship_validation_df = (
    table_existence_validation_df
    .unionByName(relationship_core_validation_df)
)

RELATIONSHIP_RESULTS_TABLE = f"{SILVER_SCHEMA}.silver_relationship_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {RELATIONSHIP_RESULTS_TABLE}
    WHERE validation_run_id = '{VALIDATION_RUN_ID}'
    """)
except Exception as e:
    print("Relationship validation results table does not exist yet. It will be created now.")

(
    all_relationship_validation_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(RELATIONSHIP_RESULTS_TABLE)
)

display(
    spark.table(RELATIONSHIP_RESULTS_TABLE)
    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

final_fail_count = (
    spark.table(RELATIONSHIP_RESULTS_TABLE)
    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)
    .filter(F.col("status") == "FAIL")
    .count()
)

print("Final Silver relationship validation failures:", final_fail_count)

if final_fail_count == 0:
    print("✅ Notebook 14 PASSED: Silver relationship validation completed successfully.")
    print("Next phase: High-level Silver domain views, then Gold marts.")
else:
    print("❌ Notebook 14 FAILED: Review relationship validation failures before moving forward.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

SILVER_SCHEMA = "supplysage_silver"

VIEW_BUILD_RUN_ID = f"silver_domain_views_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "15_silver_domain_views"

print("VIEW_BUILD_RUN_ID:", VIEW_BUILD_RUN_ID)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

required_domain_input_tables = [
    "silver_m5_calendar",
    "silver_m5_sales_daily",
    "silver_m5_sell_prices",
    "silver_retail_inventory",
    "silver_dataco_orders_shipments",
    "silver_dataco_field_dictionary",
    "silver_suppliers",
    "silver_supplier_aliases",
    "silver_supplier_sku_map",
    "silver_alternate_suppliers",
    "silver_supplier_scorecards",
    "silver_purchase_orders",
    "silver_shipment_routes",
    "silver_product_crosswalk",
    "silver_external_risk_events",
    "silver_external_evidence_documents",
    "silver_transform_validation_results",
    "silver_relationship_validation_results"
]

input_table_rows = []

for table_name in required_domain_input_tables:
    full_table_name = f"{SILVER_SCHEMA}.{table_name}"

    try:
        row_count = spark.table(full_table_name).count()
        status = "FOUND"
        issue_detail = None
        print(f"{table_name}: {row_count} rows")

    except Exception as e:
        row_count = 0
        status = "MISSING"
        issue_detail = str(e)[:500]
        print(f"{table_name}: MISSING")

    input_table_rows.append({
        "table_name": table_name,
        "row_count": row_count,
        "status": status,
        "issue_detail": issue_detail
    })

input_table_schema = T.StructType([
    T.StructField("table_name", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

input_table_df = spark.createDataFrame(input_table_rows, schema=input_table_schema)

display(input_table_df.orderBy("status", "table_name"))

missing_count = input_table_df.filter(F.col("status") == "MISSING").count()

print("Missing input tables:", missing_count)

if missing_count == 0:
    print("✅ Chunk 1 PASSED: All Silver inputs are available for domain view creation.")
else:
    print("❌ Chunk 1 FAILED: Some Silver inputs are missing.")

In [0]:
# Notebook 15, Chunk 2: Create 7 high-level Silver domain views

# 1. Calendar domain view
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_calendar AS
SELECT
    calendar_date_id,
    calendar_date,
    m5_day_id,
    wm_year_week,
    weekday_name,
    weekday_number,
    calendar_year,
    calendar_quarter,
    month_number,
    day_of_month,
    week_of_year,
    is_weekend,
    is_event_day,
    event_count,
    event_name_1,
    event_type_1,
    event_name_2,
    event_type_2,
    snap_ca,
    snap_tx,
    snap_wi,
    is_snap_any_state
FROM {SILVER_SCHEMA}.silver_m5_calendar
""")


# 2. Retail sales + price daily domain view
# Grain: item/store/day.
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_retail_sales_price_daily AS
SELECT
    s.sales_record_id,
    s.m5_series_id,
    s.item_id,
    s.department_id,
    s.category_id,
    s.store_id,
    s.state_id,
    s.m5_day_id,
    s.calendar_date_id,
    s.calendar_date,
    s.calendar_year,
    s.month_number,
    s.wm_year_week,
    s.weekday_name,
    s.is_weekend,
    s.is_event_day,
    s.event_count,
    s.is_snap_any_state,
    s.units_sold,
    p.sell_price,
    CASE 
        WHEN p.sell_price IS NOT NULL THEN s.units_sold * p.sell_price
        ELSE NULL
    END AS estimated_sales_amount,
    p.is_valid_price
FROM {SILVER_SCHEMA}.silver_m5_sales_daily s
LEFT JOIN {SILVER_SCHEMA}.silver_m5_sell_prices p
    ON s.store_id = p.store_id
   AND s.item_id = p.item_id
   AND s.wm_year_week = p.wm_year_week
""")


# 3. Inventory / stockout snapshot domain view
# Grain: inventory_date + store + product.
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_inventory_stockout_snapshot AS
SELECT
    inv.inventory_record_id,
    inv.inventory_date_id,
    inv.inventory_date,
    inv.store_id,
    inv.product_id,
    inv.category,
    inv.region,
    inv.inventory_level,
    inv.units_sold,
    inv.units_ordered,
    inv.demand_forecast_raw,
    inv.demand_forecast,
    inv.demand_forecast_was_negative,
    inv.price,
    inv.discount,
    inv.weather_condition,
    inv.holiday_promotion_flag,
    inv.competitor_pricing,
    inv.seasonality,
    inv.inventory_position,
    inv.stockout_gap_units,
    inv.is_stockout_risk,
    inv.inventory_coverage_ratio,
    inv.sell_through_rate,
    inv.price_vs_competitor,
    cw.canonical_sku_id,
    cw.confidence AS product_mapping_confidence,
    cw.mapping_rule AS product_mapping_rule
FROM {SILVER_SCHEMA}.silver_retail_inventory inv
LEFT JOIN {SILVER_SCHEMA}.silver_product_crosswalk cw
    ON inv.product_id = cw.source_product_id
   AND lower(cw.source_system) LIKE '%retail%'
""")


# 4. Orders / shipments / logistics domain view
# Grain: order line / shipment record.
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_orders_shipments_logistics AS
SELECT
    order_shipment_record_id,
    order_id,
    order_item_id,
    order_timestamp,
    shipping_timestamp,
    order_date_id,
    order_date,
    shipping_date_id,
    shipping_date,
    order_year,
    order_month,
    delivery_status,
    late_delivery_risk_flag,
    days_for_shipping_real,
    days_for_shipment_scheduled,
    shipping_delay_days,
    is_late_delivery,
    is_delayed_beyond_schedule,
    shipping_mode,
    order_status,
    is_cancelled_or_suspect_order,
    market,
    order_region,
    order_country,
    order_state,
    order_city,
    customer_segment,
    department_name,
    category_name,
    dataco_product_card_id,
    product_name,
    order_item_quantity,
    sales,
    order_item_total,
    order_profit_per_order,
    benefit_per_order,
    sales_per_customer,
    profit_margin
FROM {SILVER_SCHEMA}.silver_dataco_orders_shipments
""")


# 5. Supplier network domain view
# Grain: supplier.
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_supplier_network AS
WITH supplier_sku_summary AS (
    SELECT
        supplier_id,
        COUNT(DISTINCT sku_id) AS mapped_sku_count,
        SUM(CASE WHEN is_primary_supplier = true THEN 1 ELSE 0 END) AS primary_sku_count,
        SUM(CASE WHEN alternate_supplier_available = true THEN 1 ELSE 0 END) AS sku_with_alternate_count,
        AVG(dependency_percent) AS avg_dependency_percent,
        AVG(standard_lead_time_days) AS avg_standard_lead_time_days
    FROM {SILVER_SCHEMA}.silver_supplier_sku_map
    GROUP BY supplier_id
),
po_summary AS (
    SELECT
        supplier_id,
        COUNT(DISTINCT po_id) AS purchase_order_count,
        COUNT(*) AS purchase_order_line_count,
        AVG(po_fill_rate) AS avg_po_fill_rate,
        AVG(actual_lead_time_days) AS avg_actual_lead_time_days,
        AVG(delivery_delay_days) AS avg_delivery_delay_days,
        SUM(CASE WHEN is_po_late = true THEN 1 ELSE 0 END) AS late_po_line_count
    FROM {SILVER_SCHEMA}.silver_purchase_orders
    GROUP BY supplier_id
),
scorecard_latest AS (
    SELECT *
    FROM (
        SELECT
            supplier_id,
            scorecard_month,
            fill_rate,
            on_time_delivery_rate,
            quality_issue_rate,
            avg_lead_time_days,
            lead_time_variance,
            defect_rate,
            is_deteriorating,
            ROW_NUMBER() OVER (
                PARTITION BY supplier_id
                ORDER BY scorecard_month DESC
            ) AS rn
        FROM {SILVER_SCHEMA}.silver_supplier_scorecards
    )
    WHERE rn = 1
),
route_summary AS (
    SELECT
        supplier_id,
        COUNT(DISTINCT route_id) AS route_count,
        COUNT(DISTINCT origin_country) AS origin_country_count,
        COUNT(DISTINCT origin_port) AS origin_port_count,
        COUNT(DISTINCT risk_region) AS risk_region_count,
        AVG(standard_transit_days) AS avg_standard_transit_days
    FROM {SILVER_SCHEMA}.silver_shipment_routes
    GROUP BY supplier_id
),
alias_summary AS (
    SELECT
        supplier_id,
        COUNT(*) AS alias_count
    FROM {SILVER_SCHEMA}.silver_supplier_aliases
    GROUP BY supplier_id
)
SELECT
    s.supplier_record_id,
    s.supplier_id,
    s.supplier_name,
    s.parent_company,
    s.country,
    s.region,
    s.supplier_category,
    s.criticality_tier,
    s.annual_spend,
    s.single_source_flag,
    s.default_lead_time_days,
    COALESCE(a.alias_count, 0) AS alias_count,
    COALESCE(ss.mapped_sku_count, 0) AS mapped_sku_count,
    COALESCE(ss.primary_sku_count, 0) AS primary_sku_count,
    COALESCE(ss.sku_with_alternate_count, 0) AS sku_with_alternate_count,
    ss.avg_dependency_percent,
    ss.avg_standard_lead_time_days,
    COALESCE(po.purchase_order_count, 0) AS purchase_order_count,
    COALESCE(po.purchase_order_line_count, 0) AS purchase_order_line_count,
    po.avg_po_fill_rate,
    po.avg_actual_lead_time_days,
    po.avg_delivery_delay_days,
    COALESCE(po.late_po_line_count, 0) AS late_po_line_count,
    sc.scorecard_month AS latest_scorecard_month,
    sc.fill_rate AS latest_fill_rate,
    sc.on_time_delivery_rate AS latest_on_time_delivery_rate,
    sc.quality_issue_rate AS latest_quality_issue_rate,
    sc.avg_lead_time_days AS latest_avg_lead_time_days,
    sc.lead_time_variance AS latest_lead_time_variance,
    sc.defect_rate AS latest_defect_rate,
    sc.is_deteriorating AS latest_is_deteriorating,
    COALESCE(r.route_count, 0) AS route_count,
    COALESCE(r.origin_country_count, 0) AS origin_country_count,
    COALESCE(r.origin_port_count, 0) AS origin_port_count,
    COALESCE(r.risk_region_count, 0) AS risk_region_count,
    r.avg_standard_transit_days,
    s.synthetic_batch_id,
    s.is_synthetic
FROM {SILVER_SCHEMA}.silver_suppliers s
LEFT JOIN alias_summary a
    ON s.supplier_id = a.supplier_id
LEFT JOIN supplier_sku_summary ss
    ON s.supplier_id = ss.supplier_id
LEFT JOIN po_summary po
    ON s.supplier_id = po.supplier_id
LEFT JOIN scorecard_latest sc
    ON s.supplier_id = sc.supplier_id
LEFT JOIN route_summary r
    ON s.supplier_id = r.supplier_id
""")


# 6. External risk events / evidence domain view
# Grain: external event.
# 6. External risk events / evidence domain view
# Grain: external event.
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_external_risk_events AS
SELECT
    e.external_event_id,
    e.source_name,
    e.risk_category,
    e.event_type,
    e.event_timestamp,
    e.event_date,
    e.event_title,
    e.event_summary,
    e.source_url,
    e.source_domain,
    e.event_country,
    e.event_region,
    e.language,
    e.severity,
    e.evidence_type,
    e.api_run_id,
    e.source_payload_hash,
    d.evidence_document_id,
    d.raw_payload_preview,
    d.raw_payload_size_chars,
    d.evidence_storage_mode,
    e.ingestion_batch_id,
    e.silver_transform_run_id
FROM {SILVER_SCHEMA}.silver_external_risk_events e
LEFT JOIN {SILVER_SCHEMA}.silver_external_evidence_documents d
    ON e.api_run_id = d.api_run_id
   AND e.source_payload_hash = d.payload_hash
""")


# 7. Pipeline governance / audit domain view
spark.sql(f"""
CREATE OR REPLACE VIEW {SILVER_SCHEMA}.silver_domain_pipeline_governance_audit AS
SELECT
    'silver_transform_validation' AS audit_source,
    transform_run_id AS run_id,
    source_table,
    target_table,
    validation_check,
    expected_value,
    actual_value,
    status,
    issue_detail,
    validated_at,
    created_by,
    notebook_name
FROM {SILVER_SCHEMA}.silver_transform_validation_results

UNION ALL

SELECT
    'silver_relationship_validation' AS audit_source,
    validation_run_id AS run_id,
    silver_table AS source_table,
    silver_table AS target_table,
    validation_check,
    expected_value,
    actual_value,
    status,
    issue_detail,
    validated_at,
    created_by,
    notebook_name
FROM {SILVER_SCHEMA}.silver_relationship_validation_results
""")


domain_views = [
    "silver_domain_calendar",
    "silver_domain_retail_sales_price_daily",
    "silver_domain_inventory_stockout_snapshot",
    "silver_domain_orders_shipments_logistics",
    "silver_domain_supplier_network",
    "silver_domain_external_risk_events",
    "silver_domain_pipeline_governance_audit"
]

print("Created Silver domain views:")

for view_name in domain_views:
    print(f"✅ {SILVER_SCHEMA}.{view_name}")

print("✅ Chunk 2 PASSED: 7 high-level Silver domain views created.")

In [0]:
# Notebook 15, Chunk 3: Validate 7 Silver domain views

domain_view_validation_rows = []

domain_views = [
    {
        "view_name": "silver_domain_calendar",
        "expected_min_rows": 1969,
        "use_full_count": True
    },
    {
        "view_name": "silver_domain_retail_sales_price_daily",
        "expected_min_rows": 1,
        "use_full_count": False
    },
    {
        "view_name": "silver_domain_inventory_stockout_snapshot",
        "expected_min_rows": 73100,
        "use_full_count": True
    },
    {
        "view_name": "silver_domain_orders_shipments_logistics",
        "expected_min_rows": 180519,
        "use_full_count": True
    },
    {
        "view_name": "silver_domain_supplier_network",
        "expected_min_rows": 48,
        "use_full_count": True
    },
    {
        "view_name": "silver_domain_external_risk_events",
        "expected_min_rows": 11508,
        "use_full_count": True
    },
    {
        "view_name": "silver_domain_pipeline_governance_audit",
        "expected_min_rows": 1,
        "use_full_count": True
    }
]

for item in domain_views:
    view_name = item["view_name"]
    expected_min_rows = item["expected_min_rows"]
    use_full_count = item["use_full_count"]
    full_view_name = f"{SILVER_SCHEMA}.{view_name}"

    try:
        df = spark.table(full_view_name)

        if use_full_count:
            actual_row_count = df.count()
        else:
            # Avoid expensive full count on the large sales + price daily view.
            actual_row_count = df.limit(100).count()

        status = "PASS" if actual_row_count >= expected_min_rows or (not use_full_count and actual_row_count > 0) else "FAIL"
        issue_detail = None if status == "PASS" else f"Expected at least {expected_min_rows} rows, found {actual_row_count}."

    except Exception as e:
        actual_row_count = 0
        status = "FAIL"
        issue_detail = str(e)[:500]

    domain_view_validation_rows.append({
        "view_build_run_id": VIEW_BUILD_RUN_ID,
        "view_name": view_name,
        "validation_check": "domain_view_queryable_and_non_empty",
        "expected_value": str(expected_min_rows),
        "actual_value": str(actual_row_count),
        "status": status,
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


domain_view_validation_schema = T.StructType([
    T.StructField("view_build_run_id", T.StringType(), True),
    T.StructField("view_name", T.StringType(), True),
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True),
    T.StructField("validated_at", T.StringType(), True),
    T.StructField("created_by", T.StringType(), True),
    T.StructField("notebook_name", T.StringType(), True)
])

domain_view_validation_df = spark.createDataFrame(
    domain_view_validation_rows,
    schema=domain_view_validation_schema
)

display(domain_view_validation_df.orderBy("status", "view_name"))

domain_view_fail_count = domain_view_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver domain view validation failures:", domain_view_fail_count)

In [0]:
# Notebook 15, Chunk 4: Save Silver domain view validation results

DOMAIN_VIEW_RESULTS_TABLE = f"{SILVER_SCHEMA}.silver_domain_view_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {DOMAIN_VIEW_RESULTS_TABLE}
    WHERE view_build_run_id = '{VIEW_BUILD_RUN_ID}'
    """)
except Exception as e:
    print("Domain view validation results table does not exist yet. It will be created now.")

(
    domain_view_validation_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DOMAIN_VIEW_RESULTS_TABLE)
)

display(
    spark.table(DOMAIN_VIEW_RESULTS_TABLE)
    .filter(F.col("view_build_run_id") == VIEW_BUILD_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

final_fail_count = (
    spark.table(DOMAIN_VIEW_RESULTS_TABLE)
    .filter(F.col("view_build_run_id") == VIEW_BUILD_RUN_ID)
    .filter(F.col("status") == "FAIL")
    .count()
)

print("Final Silver domain view validation failures:", final_fail_count)

if final_fail_count == 0:
    print("✅ Notebook 15 PASSED: 7 high-level Silver domain views created and validated.")
    print("Next phase: Gold marts.")
else:
    print("❌ Notebook 15 FAILED: Review failed domain view checks before moving to Gold.")